# Model Building — Malware

Notebook atualizado da fase de **Model Building, Evaluation e Optimization**.

Fluxo:
1. Importação direta do Kaggle
2. Data Preparation
3. Split 70/15/15
4. Decision Tree, Random Forest e XGBoost
5. Validação e cross-validation
6. RandomizedSearchCV + GridSearchCV + GridSearchCV refinado
7. Comparação e seleção final
8. Teste reservado
9. ROC-AUC, overfitting e gráficos
10. Persistência do modelo e das features


## 1. Preparação do ambiente


In [ ]:
%pip install -q kagglehub[pandas-datasets]
%pip install -q xgboost
%pip install -q matplotlib
%pip install -q scikit-learn


In [ ]:
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    if not Path("IAAC-Projeto").exists():
        !git clone https://github.com/diogocarvalho2025056-crypto/IAAC-Projeto.git
    pasta_projeto = Path("IAAC-Projeto").resolve()
else:
    pasta_projeto = Path.cwd().parent.resolve()

if str(pasta_projeto) not in sys.path:
    sys.path.insert(0, str(pasta_projeto))

print("Pasta do projeto:", pasta_projeto)


## 2. Imports


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
import kagglehub

from kagglehub import KaggleDatasetAdapter

from src.data_preparation_utils import (
    remover_colunas,
    remover_duplicados,
    separar_features_target,
    dividir_treino_validacao_teste
)

from src.model_utils import (
    criar_decision_tree,
    criar_random_forest,
    criar_xgboost,
    treinar_modelo,
    prever,
    avaliar_modelo,
    comparar_modelos,
    importancia_features,
    avaliar_cross_validation,
    selecionar_melhor_modelo,
    obter_modelo_selecionado,
    otimizar_xgboost,
    otimizar_xgboost_random,
    otimizar_xgboost_refinado,
    calcular_roc_auc,
    analisar_overfitting
)


## 3. Importação direta do Kaggle


In [ ]:
file_path = "Malware_and_benign_recognition.csv"

df = kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "aashishzzz/malware-detection-dataset-with-new-malware-info",
    file_path
)

print("Dimensão original:", df.shape)
df.head()


## 4. Data Preparation


In [ ]:
df_preparado = remover_colunas(
    df.copy(),
    ["File", "SuspiciousSectionNames", "SectionCount"]
)

df_preparado = remover_duplicados(df_preparado)

X, y = separar_features_target(
    df_preparado,
    "Malicious"
)

print("Dataset preparado:", df_preparado.shape)
print("X:", X.shape)
print("y:", y.shape)


## 5. Split treino / validação / teste


In [ ]:
(
    X_train,
    X_val,
    X_test,
    y_train,
    y_val,
    y_test
) = dividir_treino_validacao_teste(X, y)

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_val:", y_val.shape)
print("y_test:", y_test.shape)


## 6. Treino dos modelos baseline


In [ ]:
decision_tree = treinar_modelo(
    criar_decision_tree(),
    X_train,
    y_train
)

random_forest = treinar_modelo(
    criar_random_forest(),
    X_train,
    y_train
)

xgboost = treinar_modelo(
    criar_xgboost(),
    X_train,
    y_train
)


## 7. Avaliação na validação


In [ ]:
y_pred_val_dt = prever(decision_tree, X_val)
y_pred_val_rf = prever(random_forest, X_val)
y_pred_val_xgb = prever(xgboost, X_val)

resultado_dt, matriz_dt, relatorio_dt = avaliar_modelo(
    "Decision Tree", y_val, y_pred_val_dt, ["Benigno", "Malware"]
)

resultado_rf, matriz_rf, relatorio_rf = avaliar_modelo(
    "Random Forest", y_val, y_pred_val_rf, ["Benigno", "Malware"]
)

resultado_xgb, matriz_xgb, relatorio_xgb = avaliar_modelo(
    "XGBoost", y_val, y_pred_val_xgb, ["Benigno", "Malware"]
)

resultados_validacao = [
    resultado_dt,
    resultado_rf,
    resultado_xgb
]

tabela_validacao = comparar_modelos(resultados_validacao)
tabela_validacao


In [ ]:
for nome, matriz, relatorio in [
    ("Decision Tree", matriz_dt, relatorio_dt),
    ("Random Forest", matriz_rf, relatorio_rf),
    ("XGBoost", matriz_xgb, relatorio_xgb)
]:
    print("\n", nome)
    print("Matriz de confusão:")
    print(matriz)
    print(relatorio)


Os três modelos apresentam o mesmo desempenho na validação. Por isso, a seleção é aprofundada com **5-Fold Cross-Validation**.


## 8. Importância das features


In [ ]:
for nome, modelo in [
    ("Decision Tree", decision_tree),
    ("Random Forest", random_forest),
    ("XGBoost", xgboost)
]:
    print(f"\n--- {nome} ---")
    display(importancia_features(modelo, X.columns).head(10))


## 9. Cross-validation


In [ ]:
resultado_cv_dt, recall_dt, f1_dt = avaliar_cross_validation(
    "Decision Tree", decision_tree, X_train, y_train
)

resultado_cv_rf, recall_rf, f1_rf = avaliar_cross_validation(
    "Random Forest", random_forest, X_train, y_train
)

resultado_cv_xgb, recall_xgb, f1_xgb = avaliar_cross_validation(
    "XGBoost", xgboost, X_train, y_train
)

resultados_cv = [
    resultado_cv_dt,
    resultado_cv_rf,
    resultado_cv_xgb
]

tabela_cv = pd.DataFrame(resultados_cv)
tabela_cv


## 10. Seleção do melhor baseline


In [ ]:
nome_melhor_modelo = selecionar_melhor_modelo(
    resultados_validacao,
    resultados_cv
)

melhor_modelo = obter_modelo_selecionado(
    nome_melhor_modelo,
    decision_tree,
    random_forest,
    xgboost
)

print("Melhor modelo baseline:", nome_melhor_modelo)


## 11. RandomizedSearchCV


In [ ]:
random_search = otimizar_xgboost_random(
    X_train,
    y_train
)

print("Melhores parâmetros:")
print(random_search.best_params_)
print("Melhor Recall médio CV:", random_search.best_score_)


## 12. GridSearchCV original


In [ ]:
grid_search = otimizar_xgboost(
    X_train,
    y_train
)

print("Melhores parâmetros:")
print(grid_search.best_params_)
print("Melhor Recall médio CV:", grid_search.best_score_)


## 13. GridSearchCV refinado


In [ ]:
grid_refinado = otimizar_xgboost_refinado(
    X_train,
    y_train
)

print("Melhores parâmetros refinados:")
print(grid_refinado.best_params_)
print("Melhor Recall médio CV:", grid_refinado.best_score_)


As três estratégias atingiram o mesmo melhor Recall médio em cross-validation, cerca de **99,75%**.


## 14. Comparação das versões do XGBoost


In [ ]:
modelo_grid_original = grid_search.best_estimator_
modelo_random = random_search.best_estimator_
modelo_refinado = grid_refinado.best_estimator_

resultado_grid_original, _, _ = avaliar_modelo(
    "XGBoost GridSearch Original",
    y_val,
    prever(modelo_grid_original, X_val),
    ["Benigno", "Malware"]
)

resultado_random, _, _ = avaliar_modelo(
    "XGBoost RandomizedSearch",
    y_val,
    prever(modelo_random, X_val),
    ["Benigno", "Malware"]
)

resultado_refinado, _, _ = avaliar_modelo(
    "XGBoost GridSearch Refinado",
    y_val,
    prever(modelo_refinado, X_val),
    ["Benigno", "Malware"]
)

comparacao_tuning = pd.DataFrame([
    resultado_xgb,
    resultado_grid_original,
    resultado_random,
    resultado_refinado
])

comparacao_tuning


As quatro versões apresentam as mesmas métricas de validação. Pelo princípio **Keep it simple**, o baseline é mantido se houver empate.


## 15. Seleção final


In [ ]:
candidatos_xgboost = [
    ("XGBoost Baseline", xgboost, resultado_xgb),
    ("XGBoost GridSearch Original", modelo_grid_original, resultado_grid_original),
    ("XGBoost RandomizedSearch", modelo_random, resultado_random),
    ("XGBoost GridSearch Refinado", modelo_refinado, resultado_refinado)
]

def criterio_selecao(item):
    resultado = item[2]
    return (
        resultado["Recall"],
        resultado["F1-score"],
        resultado["Accuracy"]
    )

nome_modelo_final, modelo_final, resultado_validacao_final = max(
    candidatos_xgboost,
    key=criterio_selecao
)

print("Modelo final selecionado:", nome_modelo_final)


## 16. Overfitting


In [ ]:
comparacao_overfitting = pd.DataFrame([
    analisar_overfitting(
        "Decision Tree",
        decision_tree,
        X_train, y_train,
        X_val, y_val
    ),
    analisar_overfitting(
        "Random Forest",
        random_forest,
        X_train, y_train,
        X_val, y_val
    ),
    analisar_overfitting(
        "XGBoost Baseline",
        xgboost,
        X_train, y_train,
        X_val, y_val
    ),
    analisar_overfitting(
        "XGBoost GridSearch Original",
        modelo_grid_original,
        X_train, y_train,
        X_val, y_val
    ),
    analisar_overfitting(
        "XGBoost RandomizedSearch",
        modelo_random,
        X_train, y_train,
        X_val, y_val
    ),
    analisar_overfitting(
        "XGBoost GridSearch Refinado",
        modelo_refinado,
        X_train, y_train,
        X_val, y_val
    )
])

comparacao_overfitting


As versões otimizadas reduzem o gap treino-validação, mas não melhoram o desempenho de validação. Isso não justifica substituir o baseline.


## 17. Avaliação final no teste


In [ ]:
y_pred_test = prever(
    modelo_final,
    X_test
)

resultado_final, matriz_final, relatorio_final = avaliar_modelo(
    nome_modelo_final + " - Teste Final",
    y_test,
    y_pred_test,
    ["Benigno", "Malware"]
)

print(resultado_final)
print("\nMatriz de confusão:")
print(matriz_final)
print("\nRelatório de classificação:")
print(relatorio_final)


Resultado esperado com a versão atual:

- Accuracy ≈ **97,69%**
- Precision ≈ **95,51%**
- Recall = **100%**
- F1-score ≈ **97,70%**
- 0 falsos negativos


## 18. ROC-AUC


In [ ]:
auc, fpr, tpr, thresholds = calcular_roc_auc(
    modelo_final,
    X_test,
    y_test
)

print("AUC:", auc)

plt.figure(figsize=(8, 6))
plt.plot(
    fpr,
    tpr,
    label=f"{nome_modelo_final} - AUC = {auc:.4f}"
)
plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
    label="Classificador aleatório"
)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Modelo Final")
plt.legend()
plt.tight_layout()
plt.show()


## 19. Gráficos comparativos


In [ ]:
accuracy_modelos = tabela_validacao[
    ["Modelo", "Accuracy"]
].copy()

plt.figure(figsize=(8, 5))
plt.bar(
    accuracy_modelos["Modelo"],
    accuracy_modelos["Accuracy"]
)
plt.title("Accuracy dos Modelos - Validação")
plt.xlabel("Modelo")
plt.ylabel("Accuracy")
plt.ylim(0.95, 1.00)
plt.tight_layout()
plt.show()


In [ ]:
metricas_validacao = tabela_validacao.set_index("Modelo")[
    ["Accuracy", "Precision", "Recall", "F1-score"]
]

metricas_validacao.plot(
    kind="bar",
    figsize=(10, 6)
)
plt.title("Comparação das Métricas dos Modelos - Validação")
plt.xlabel("Modelo")
plt.ylabel("Resultado")
plt.ylim(0.95, 1.00)
plt.xticks(rotation=0)
plt.legend(title="Métrica")
plt.tight_layout()
plt.show()


In [ ]:
metricas_cv = tabela_cv.set_index("Modelo")[
    ["Recall CV", "F1 CV"]
]

metricas_cv.plot(
    kind="bar",
    figsize=(9, 6)
)
plt.title("Comparação dos Modelos - 5-Fold Cross-Validation")
plt.xlabel("Modelo")
plt.ylabel("Resultado médio")
plt.ylim(0.97, 1.00)
plt.xticks(rotation=0)
plt.legend(title="Métrica")
plt.tight_layout()
plt.show()


In [ ]:
metricas_tuning = comparacao_tuning.set_index("Modelo")[
    ["Accuracy", "Precision", "Recall", "F1-score"]
]

metricas_tuning.plot(
    kind="bar",
    figsize=(12, 6)
)
plt.title("Comparação das Versões do XGBoost")
plt.xlabel("Modelo")
plt.ylabel("Resultado")
plt.ylim(0.95, 1.00)
plt.xticks(rotation=15)
plt.legend(title="Métrica")
plt.tight_layout()
plt.show()


In [ ]:
dados_overfitting = comparacao_overfitting.set_index("Modelo")[
    ["Accuracy treino", "Accuracy validação"]
]

dados_overfitting.plot(
    kind="bar",
    figsize=(12, 6)
)
plt.title("Accuracy - Treino vs Validação")
plt.xlabel("Modelo")
plt.ylabel("Accuracy")
plt.ylim(0.95, 1.01)
plt.xticks(rotation=20)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
importancias_finais = importancia_features(
    modelo_final,
    X.columns
).head(10)

importancias_finais = importancias_finais.sort_values(
    by="Importancia"
)

plt.figure(figsize=(9, 6))
plt.barh(
    importancias_finais["Feature"],
    importancias_finais["Importancia"]
)
plt.title(f"Top 10 Features - {nome_modelo_final}")
plt.xlabel("Importância")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


## 20. Guardar o modelo final e as features


In [ ]:
pasta_models = pasta_projeto / "models"
pasta_models.mkdir(exist_ok=True)

caminho_modelo = pasta_models / "malware_xgboost.json"
modelo_final.save_model(caminho_modelo)

features = list(X.columns)
caminho_features = pasta_models / "malware_features.json"

with open(
    caminho_features,
    "w",
    encoding="utf-8"
) as ficheiro:
    json.dump(
        features,
        ficheiro,
        indent=4,
        ensure_ascii=False
    )

print("Modelo guardado em:", caminho_modelo)
print("Features guardadas em:", caminho_features)


## 21. Conclusão

A cross-validation selecionou o XGBoost como melhor baseline. Foram depois testadas três estratégias de tuning, mas nenhuma melhorou as métricas de validação. Assim, manteve-se o **XGBoost Baseline**.

O modelo final é avaliado uma única vez no conjunto de teste reservado e é depois guardado, juntamente com a lista das 23 features, para utilização na fase de deployment.
